In [1]:
import torch
import numpy as np
import pandas as pd
import plotly.io as pio
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

In [2]:
DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
pio.templates.default = "plotly_white"

In [4]:
SYMBOLS = ["SBIN", "LICI", "HAL", "NTPC", "ONGC"]
START, END = "2025-01-01", "2026-01-31"
FEATURES = ["close", "MA10", "MA20", "MA50"]

In [5]:
WINDOW, HIDDEN = 60, 64
EPOCHS, BATCH_SIZE = 100, 32
LEARNING_RATE, SEED = 0.001, 42

In [6]:
DEVICE = torch.device("cpu")
torch.set_num_threads(2)
print("Device:", DEVICE)

Device: cpu


In [7]:
stocks = {}
quality = []
for symbol in SYMBOLS:
    files = sorted((DATA_DIR / symbol).glob("*/*.csv"))
    
    if not files:
        raise FileNotFoundError(f"No monthly CSVs in {DATA_DIR / symbol}")
    
    df = pd.concat([pd.read_csv(file) for file in files], ignore_index=True)
    
    df["date"] = pd.to_datetime(df["datetime"], utc=True).dt.tz_convert("Asia/Kolkata").dt.tz_localize(None).dt.normalize()
    
    df = df.loc[df["date"].between(START, END), ["date", "open", "high", "low", "close", "volume"]].copy()
    
    duplicates = int(df["date"].duplicated().sum())
    
    df = df.sort_values("date").drop_duplicates("date", keep="last").reset_index(drop=True)
    
    for column in ["open", "high", "low", "close", "volume"]:
        df[column] = pd.to_numeric(df[column], errors="raise")
    
    if df.empty or not np.isfinite(df[["open", "high", "low", "close", "volume"]].to_numpy()).all():
        raise ValueError(f"{symbol}: empty dataset or missing/non-finite OHLCV")
    
    bad = (df[["open", "high", "low", "close"]].le(0).any(axis=1)
           | df["volume"].lt(0)
           | df["high"].lt(df[["open", "low", "close"]].max(axis=1))
           | df["low"].gt(df[["open", "high", "close"]].min(axis=1)))
    
    if bad.any():
        raise ValueError(f"{symbol}: {int(bad.sum())} invalid OHLCV rows; inspect your CSVs")
    
    for days in [10, 20, 50]:
        df[f"MA{days}"] = df["close"].rolling(days).mean()
    
    df["return_pct"] = df["close"].pct_change(fill_method=None) * 100
    
    stocks[symbol] = df
    
    quality.append({
        "symbol":symbol, 
        "rows":len(df), 
        "logged_rows":269,
        "matches_logged_total":len(df) == 269,
        "duplicate_dates_removed":duplicates,
        "first_date":df["date"].min(), 
        "last_date":df["date"].max()
    })

display(pd.DataFrame(quality))

,symbol,rows,logged_rows,matches_logged_total,duplicate_dates_removed,first_date,last_date
0,SBIN,269,269,True,0,2025-01-01,2026-01-30
1,LICI,269,269,True,0,2025-01-01,2026-01-30
2,HAL,269,269,True,0,2025-01-01,2026-01-30
3,NTPC,269,269,True,0,2025-01-01,2026-01-30
4,ONGC,269,269,True,0,2025-01-01,2026-01-30


In [8]:
datasets = {}
split_rows = []
for symbol, source in stocks.items():
    df = source.dropna(subset=FEATURES).reset_index(drop=True)
    
    values = df[FEATURES].to_numpy(dtype=np.float32)
    
    targets = df["close"].to_numpy(dtype=np.float32)[WINDOW:]
    
    dates = df["date"].iloc[WINDOW:].to_numpy()
    
    if len(targets) < 2:
        raise ValueError(f"{symbol}: need at least {WINDOW + 51} daily rows")
    
    n_train = int(0.8 * len(targets))
    
    train_end = WINDOW + n_train
    
    x_min = values[:train_end].min(axis=0)
    
    x_range = values[:train_end].max(axis=0) - x_min
    
    x_range[x_range == 0] = 1
    
    y_min = float(targets[:n_train].min())
    
    y_range = float(targets[:n_train].max() - y_min) or 1.0
    
    scaled = (values - x_min) / x_range
    
    X = np.stack([scaled[i-WINDOW:i] for i in range(WINDOW, len(scaled))])
    
    y = (targets - y_min) / y_range
    
    datasets[symbol] = {
        "X_train":torch.tensor(X[:n_train], device=DEVICE),
        "y_train":torch.tensor(y[:n_train], device=DEVICE),
        "X_test":torch.tensor(X[n_train:], device=DEVICE),
        "actual":targets[n_train:], "dates":dates[n_train:],
        "last_close":df["close"].to_numpy()[WINDOW-1:-1][n_train:],
        "x_min":x_min, "x_range":x_range, "y_min":y_min, "y_range":y_range
    }
    
    split_rows.append({
        "symbol":symbol, 
        "raw_rows":len(source), 
        "feature_rows":len(df),
        "train_windows":n_train,
        "test_windows":len(targets)-n_train,
        "last_train_target":pd.Timestamp(dates[n_train-1]),
        "first_test_target":pd.Timestamp(dates[n_train])
    })

display(pd.DataFrame(split_rows))

,symbol,raw_rows,feature_rows,train_windows,test_windows,last_train_target,first_test_target
0,SBIN,269,220,128,32,2025-12-12,2025-12-15
1,LICI,269,220,128,32,2025-12-12,2025-12-15
2,HAL,269,220,128,32,2025-12-12,2025-12-15
3,NTPC,269,220,128,32,2025-12-12,2025-12-15
4,ONGC,269,220,128,32,2025-12-12,2025-12-15


In [9]:
models = {}
histories = []
for symbol, data in datasets.items():
    torch.manual_seed(SEED)
    
    parameters = {
        "Wx":(torch.randn(len(FEATURES), HIDDEN, device=DEVICE) / np.sqrt(len(FEATURES))).requires_grad_(),
        "Wh":(torch.randn(HIDDEN, HIDDEN, device=DEVICE) / np.sqrt(HIDDEN)).requires_grad_(),
        "bh":torch.zeros(HIDDEN, device=DEVICE, requires_grad=True),
        "Wy":(torch.randn(HIDDEN, 1, device=DEVICE) / np.sqrt(HIDDEN)).requires_grad_(),
        "by":torch.zeros(1, device=DEVICE, requires_grad=True)
    }
    
    optimizer = torch.optim.Adam(list(parameters.values()), lr=LEARNING_RATE)
    
    epoch_losses = []
    
    X_train, y_train = data["X_train"], data["y_train"]

    for epoch in range(1, EPOCHS + 1):
        total_loss = 0.0
        
        for start in range(0, len(X_train), BATCH_SIZE):
            xb = X_train[start:start+BATCH_SIZE]
            
            yb = y_train[start:start+BATCH_SIZE]
            
            optimizer.zero_grad()
            
            h = torch.zeros(len(xb), HIDDEN, device=DEVICE)
            
            for t in range(WINDOW):
                h = torch.tanh(xb[:, t] @ parameters["Wx"] + h @ parameters["Wh"] + parameters["bh"])
            
            prediction = (h @ parameters["Wy"] + parameters["by"]).squeeze(-1)
            
            loss = ((prediction - yb) ** 2).mean()
            
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(list(parameters.values()), max_norm=1.0)
            
            optimizer.step()
            
            total_loss += loss.item() * len(xb)
        
        epoch_losses.append(total_loss / len(X_train))
        
        if epoch == 1 or epoch % 10 == 0:
            print(f"{symbol} | Epoch {epoch:3d}/{EPOCHS} | MSE {epoch_losses[-1]:.6f}")

    models[symbol] = {key:value.detach().clone() for key, value in parameters.items()}
    
    history = pd.DataFrame({"symbol":symbol, "epoch":range(1, EPOCHS+1), "train_mse":epoch_losses})
    
    histories.append(history)
    
    history.to_csv(OUTPUT_DIR / f"{symbol}_training.csv", index=False)
    
    torch.save({
            "weights":{key:value.cpu() for key, value in models[symbol].items()},
            "features":FEATURES, 
            "window":WINDOW, 
            "hidden":HIDDEN,
            "x_min":torch.tensor(data["x_min"]), 
            "x_range":torch.tensor(data["x_range"]),
            "y_min":data["y_min"], 
            "y_range":data["y_range"],
            "epochs":EPOCHS, 
            "seed":SEED
        }, 
        OUTPUT_DIR / f"{symbol}_rnn.pt"
    )

training = pd.concat(histories, ignore_index=True)

SBIN | Epoch   1/100 | MSE 0.095604
SBIN | Epoch  10/100 | MSE 0.005301
SBIN | Epoch  20/100 | MSE 0.001868
SBIN | Epoch  30/100 | MSE 0.001531
SBIN | Epoch  40/100 | MSE 0.001402
SBIN | Epoch  50/100 | MSE 0.001351
SBIN | Epoch  60/100 | MSE 0.001317
SBIN | Epoch  70/100 | MSE 0.001291
SBIN | Epoch  80/100 | MSE 0.001267
SBIN | Epoch  90/100 | MSE 0.001245
SBIN | Epoch 100/100 | MSE 0.001224
LICI | Epoch   1/100 | MSE 0.145189
LICI | Epoch  10/100 | MSE 0.009704
LICI | Epoch  20/100 | MSE 0.007795
LICI | Epoch  30/100 | MSE 0.007057
LICI | Epoch  40/100 | MSE 0.006656
LICI | Epoch  50/100 | MSE 0.006388
LICI | Epoch  60/100 | MSE 0.006201
LICI | Epoch  70/100 | MSE 0.006060
LICI | Epoch  80/100 | MSE 0.005948
LICI | Epoch  90/100 | MSE 0.005854
LICI | Epoch 100/100 | MSE 0.005770
HAL | Epoch   1/100 | MSE 0.193305
HAL | Epoch  10/100 | MSE 0.017185
HAL | Epoch  20/100 | MSE 0.011064
HAL | Epoch  30/100 | MSE 0.009642
HAL | Epoch  40/100 | MSE 0.008856
HAL | Epoch  50/100 | MSE 0.00833

In [10]:
fig = px.line(
    training, 
    x="epoch", 
    y="train_mse", 
    color="symbol",
    title="Training Loss", 
    labels={"epoch":"Epoch", "train_mse":"Normalized MSE"},
    width=1000, 
    height=450,
    render_mode="svg"
)
fig.show()
fig.write_image("../images/training-loss.png", width=1000, height=450, scale=2)

In [11]:
forecasts = {}
metric_rows = []
for symbol, data in datasets.items():
    p = models[symbol]
    
    with torch.no_grad():
        h = torch.zeros(len(data["X_test"]), HIDDEN, device=DEVICE)
        
        for t in range(WINDOW):
            h = torch.tanh(data["X_test"][:, t] @ p["Wx"] + h @ p["Wh"] + p["bh"])
        
        scaled_prediction = (h @ p["Wy"] + p["by"]).squeeze(-1).cpu().numpy()
    
    prediction = scaled_prediction * data["y_range"] + data["y_min"]
    
    forecast = pd.DataFrame({
        "date":data["dates"], 
        "actual":data["actual"],
        "predicted":prediction, 
        "last_close":data["last_close"]
    })
    
    forecasts[symbol] = forecast
    
    forecast.to_csv(OUTPUT_DIR / f"{symbol}_test_predictions.csv", index=False)
    
    for name, estimated in [("RNN", prediction), ("Last close", data["last_close"])]:
        errors = estimated.astype(np.float64) - data["actual"].astype(np.float64)
        
        mse = np.mean(errors ** 2)
        
        metric_rows.append({
            "symbol":symbol, 
            "model":name, 
            "MSE":mse,
            "MAE":np.mean(np.abs(errors)), 
            "RMSE":np.sqrt(mse),
            "MAPE (%)":100 * np.mean(np.abs(errors / data["actual"]))
        })

metrics = pd.DataFrame(metric_rows)
metrics.to_csv(OUTPUT_DIR / "test_metrics.csv", index=False)
display(metrics.round(4))

,symbol,model,MSE,MAE,RMSE,MAPE (%)
0,SBIN,RNN,1285.8933,28.6432,35.8594,2.7775
1,SBIN,Last close,104.5981,8.2406,10.2273,0.8103
2,LICI,RNN,72.7940,7.2320,8.5319,1.7486
3,LICI,Last close,14.3247,3.0391,3.7848,0.7284
4,HAL,RNN,7017.5565,61.2190,83.7709,1.3921
5,HAL,Last close,5414.7567,52.1656,73.5850,1.1785
6,NTPC,RNN,39.4334,4.5058,6.2796,1.3166
7,NTPC,Last close,26.2370,3.6359,5.1222,1.0640
8,ONGC,RNN,47.5058,4.5342,6.8924,1.8069
9,ONGC,Last close,25.5320,3.4125,5.0529,1.3759


In [12]:
for symbol, forecast in forecasts.items():
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(x=forecast["date"], y=forecast["actual"], name="Actual", mode="lines+markers"))
    
    fig.add_trace(go.Scatter(x=forecast["date"], y=forecast["predicted"], name="RNN predicted", mode="lines+markers"))
    
    fig.update_layout(
        title=f"{symbol}: Actual Versus Predicted Test Closes", 
        xaxis_title="Test date",
        yaxis_title="Closing price (INR)", 
        width=1000,
        height=450, 
        hovermode="x unified"
    )
    
    fig.show()
    
    fig.write_image(f"../images/{symbol}-actual-vs-predicted.png", width=1000, height=450, scale=2)

In [13]:
metric_plot = metrics.melt(id_vars=["symbol", "model"], var_name="metric", value_name="value")

fig = px.bar(
    metric_plot, 
    x="symbol", 
    y="value", 
    color="model", 
    barmode="group",
    facet_col="metric", 
    facet_col_wrap=2, 
    title="Test Metrics: RNN Versus Last Close Baseline",
    width=1000,
    height=650
)

fig.update_yaxes(matches=None)
fig.show()
fig.write_image("../images/rnn-vs-last-close.png", width=1000, height=650, scale=2)